# Grey Market Value Analysis

Which luxury watches hold their value best on the secondary market?

**Data sources:**
- eBay sold listings (actual transaction prices with sale dates)
- Chrono24 asking prices (dealer/private seller asking prices)

**Key metrics:**
- Price trends over time (appreciating vs depreciating)
- Price stability (coefficient of variation)
- Asking vs sold spread (markup over actual transaction prices)

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
from sqlalchemy import text
from watchscraper.database import engine

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.0f}'.format)

## 1. Load Data

In [ ]:
query = text("""
SELECT
    pr.id,
    b.name AS brand,
    b.slug AS brand_slug,
    w.reference_number AS ref,
    w.model_name AS model,
    w.retail_price_usd / 100.0 AS retail_price,
    w.case_material AS material,
    pr.price_usd / 100.0 AS market_price,
    pr.price_type,
    pr.condition,
    s.name AS source,
    pr.observed_at,
    pr.scraped_at
FROM price_records pr
JOIN watches w ON pr.watch_id = w.id
JOIN brands b ON w.brand_id = b.id
JOIN sources s ON pr.source_id = s.id
WHERE pr.watch_id IS NOT NULL
ORDER BY pr.observed_at NULLS LAST
""")

df = pd.read_sql(query, engine)
df['observed_at'] = pd.to_datetime(df['observed_at'], utc=True)

sold = df[df['price_type'] == 'sold'].copy()
asking = df[df['price_type'] == 'asking'].copy()

print(f'Total linked records: {len(df)}')
print(f'  Sold (eBay):    {len(sold)} — dates {sold["observed_at"].min():%Y-%m-%d} to {sold["observed_at"].max():%Y-%m-%d}')
print(f'  Asking (C24):   {len(asking)}')
print(f'  Brands: {df["brand"].nunique()}, Models: {df["ref"].nunique()}')

## 2. Grey Market Prices by Brand

In [ ]:
brand_summary = (
    sold.groupby('brand')
    .agg(
        median_price=('market_price', 'median'),
        mean_price=('market_price', 'mean'),
        min_price=('market_price', 'min'),
        max_price=('market_price', 'max'),
        sales=('id', 'count'),
        models=('ref', 'nunique'),
    )
    .sort_values('median_price', ascending=False)
)
brand_summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=sold, x='brand', y='market_price',
    order=brand_summary.index,
    palette='viridis', showfliers=False, ax=ax,
)
ax.set_ylabel('Sold Price ($)')
ax.set_xlabel('')
ax.set_title('Grey Market Price Distribution by Brand (eBay Sold)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. Price Trends Over Time

In [ ]:
# Monthly median sold price for top models by volume
sold['month'] = sold['observed_at'].dt.to_period('M')

top_models = (
    sold.groupby('ref').size()
    .sort_values(ascending=False)
    .head(12)
    .index.tolist()
)

monthly = (
    sold[sold['ref'].isin(top_models)]
    .groupby(['ref', 'brand', 'model', 'month'])
    .agg(median_price=('market_price', 'median'), sales=('id', 'count'))
    .reset_index()
)
monthly['date'] = monthly['month'].dt.to_timestamp()
monthly['label'] = monthly['brand'] + ' ' + monthly['ref']

fig = px.line(
    monthly, x='date', y='median_price', color='label',
    title='Monthly Median Sold Price — Top 12 Models by Volume',
    labels={'median_price': 'Median Sold Price ($)', 'date': 'Month'},
    hover_data=['model', 'sales'],
)
fig.update_layout(height=600, yaxis_tickformat='$,.0f')
fig.show()

## 4. Appreciating vs Depreciating (6-Month Trend)

In [ ]:
from datetime import timedelta

cutoff = sold['observed_at'].max() - timedelta(days=180)
recent = sold[sold['observed_at'] >= cutoff]
older = sold[sold['observed_at'] < cutoff]

recent_med = recent.groupby('ref')['market_price'].median().rename('recent')
older_med = older.groupby('ref')['market_price'].median().rename('older')
trend = pd.concat([recent_med, older_med], axis=1).dropna()
trend['change_pct'] = ((trend['recent'] - trend['older']) / trend['older']) * 100

# Add metadata
ref_info = sold.drop_duplicates('ref')[['ref', 'brand', 'model']].set_index('ref')
recent_counts = recent.groupby('ref').size().rename('recent_count')
trend = trend.join(ref_info).join(recent_counts)
trend = trend[trend['recent_count'] >= 3].sort_values('change_pct', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(14, max(6, len(trend) * 0.35)))
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in trend['change_pct']]
labels = [f"{row['brand']} {ref}" for ref, row in trend.iterrows()]
ax.barh(labels, trend['change_pct'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Price Change (%)')
ax.set_title(f'6-Month Grey Market Price Change (3+ recent sales)')
for i, (ref, row) in enumerate(trend.iterrows()):
    ax.text(
        row['change_pct'] + (0.5 if row['change_pct'] >= 0 else -0.5),
        i, f"${row['recent']:,.0f}",
        va='center', ha='left' if row['change_pct'] >= 0 else 'right',
        fontsize=9,
    )
plt.tight_layout()
plt.show()

print(f'\nAppreciating: {(trend["change_pct"] > 0).sum()} models')
print(f'Depreciating: {(trend["change_pct"] <= 0).sum()} models')

## 5. Price Stability (Which Watches Hold Steady?)

In [ ]:
stability = (
    sold.groupby(['brand', 'ref', 'model'])
    .agg(
        median_price=('market_price', 'median'),
        std_price=('market_price', 'std'),
        count=('id', 'count'),
    )
    .reset_index()
    .query('count >= 5')
)
stability['cv_pct'] = (stability['std_price'] / stability['median_price']) * 100
stability = stability.sort_values('cv_pct')

print('Most stable grey market prices (lowest price variance):')
print(stability[['brand', 'ref', 'model', 'median_price', 'cv_pct', 'count']].head(20).to_string(index=False))

print('\nMost volatile grey market prices:')
print(stability[['brand', 'ref', 'model', 'median_price', 'cv_pct', 'count']].tail(10).to_string(index=False))

## 6. Asking vs Sold Spread

In [ ]:
ask_med = asking.groupby('ref')['market_price'].median().rename('asking')
sold_med = sold.groupby('ref')['market_price'].median().rename('sold')
spread = pd.concat([ask_med, sold_med], axis=1).dropna()
spread['spread_pct'] = ((spread['asking'] - spread['sold']) / spread['sold']) * 100
spread = spread.join(ref_info)
spread = spread.sort_values('spread_pct', ascending=False)

fig, ax = plt.subplots(figsize=(14, max(6, len(spread) * 0.35)))
labels = [f"{row.get('brand', '?')} {ref}" for ref, row in spread.iterrows()]
ax.barh(labels, spread['spread_pct'], color=sns.color_palette('RdYlGn_r', len(spread)))
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Asking Premium Over Sold (%)')
ax.set_title('Chrono24 Asking vs eBay Sold Price Spread')
plt.tight_layout()
plt.show()

print(f'\nMedian spread across all models: {spread["spread_pct"].median():.1f}%')
print(f'Mean spread: {spread["spread_pct"].mean():.1f}%')

## 7. Price by Brand Over Time

In [ ]:
# Monthly median by brand
brand_monthly = (
    sold.groupby(['brand', 'month'])
    .agg(median_price=('market_price', 'median'), sales=('id', 'count'))
    .reset_index()
)
brand_monthly['date'] = brand_monthly['month'].dt.to_timestamp()

# Only show brands with enough data
big_brands = sold.groupby('brand').size().sort_values(ascending=False).head(5).index
brand_monthly_filt = brand_monthly[brand_monthly['brand'].isin(big_brands)]

fig = px.line(
    brand_monthly_filt, x='date', y='median_price', color='brand',
    title='Monthly Median Sold Price by Brand (Top 5 by Volume)',
    labels={'median_price': 'Median Sold Price ($)', 'date': 'Month'},
    hover_data=['sales'],
)
fig.update_layout(height=500, yaxis_tickformat='$,.0f')
fig.show()

## 8. Export Summary

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

summary = (
    sold.groupby(['brand', 'ref', 'model'])
    .agg(
        median_sold=('market_price', 'median'),
        mean_sold=('market_price', 'mean'),
        min_sold=('market_price', 'min'),
        max_sold=('market_price', 'max'),
        std_sold=('market_price', 'std'),
        sales=('id', 'count'),
        first_sale=('observed_at', 'min'),
        last_sale=('observed_at', 'max'),
    )
    .reset_index()
    .sort_values(['brand', 'median_sold'], ascending=[True, False])
)
summary['cv_pct'] = (summary['std_sold'] / summary['median_sold']) * 100

summary.to_csv('../data/grey_market_summary.csv', index=False)
print(f'Saved {len(summary)} models to data/grey_market_summary.csv')
summary.head(20)